In [2]:
import pandas as pd 
df = pd.read_csv("/lstr/sahara/datalab-ml/ibrahim/limagents_update/data/final_balanced_kde/data_gpt_kg_triplets_final_novelty_gt_600_rows_gpt_gemini_llama_balanced_data/df_kg_tripltes_novelty_gt_gpt_gemini_llama_600_balacned_data.csv") 


In [3]:
# counting token 
import pandas as pd
import tiktoken
import json

# 1. Initialize the encoder for gpt-4o-mini
# Note: gpt-4o and gpt-4o-mini use the 'o200k_base' encoding
encoding = tiktoken.get_encoding("o200k_base")

def count_tokens(text):
    if pd.isna(text):
        return 0
    # Encode the text and return the number of tokens
    return len(encoding.encode(str(text)))

# 2. Prepare your "List of Lists" as requested in the previous step
# This simulates what you are actually sending to the LLM
limitations_list = df['novelty_check_gpt'].dropna().tolist()

# 3. Calculate Tokens
# Individual row counts
row_token_counts = [count_tokens(lim) for lim in limitations_list]

# Total tokens for the content alone
total_content_tokens = sum(row_token_counts)

# Calculate overhead for the JSON structure and Prompt
# (Estimating ~200 tokens for instructions and JSON formatting)
prompt_overhead = 200 
final_estimate = total_content_tokens + prompt_overhead

# 4. Results
print(f"--- Token Usage Report ---")
print(f"Total Rows: {len(limitations_list)}")
print(f"Total Tokens (Content): {total_content_tokens:,}")
print(f"Estimated Total (Prompt + Data): {final_estimate:,}")
print(f"Average Tokens per Row: {total_content_tokens / len(limitations_list):.1f}")

# 5. Safety Check
MAX_WINDOW = 128000
if final_estimate > MAX_WINDOW:
    print(f"\n⚠️ WARNING: Your data ({final_estimate:,}) exceeds the 128k context window!")
    print("Consider processing in two smaller batches.")
else:
    print(f"\n✅ SAFE: Your data fits within the {MAX_WINDOW:,} context window.")

--- Token Usage Report ---
Total Rows: 585
Total Tokens (Content): 19,865
Estimated Total (Prompt + Data): 20,065
Average Tokens per Row: 34.0

✅ SAFE: Your data fits within the 128,000 context window.


In [ ]:
import os
import json
import pandas as pd
from openai import OpenAI

# 1. SETUP
# Use your provided key and setup
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# 2. PREPARE DATA
# We keep the data as one large list to be serialized into the prompt
limitations_list = df['novelty_check_gpt'].dropna().astype(str).tolist()

# 3. CRAFT THE CONSOLIDATED PROMPT
system_message = (
    "You are a senior research analyst. You specialize in synthesizing "
    "scientific shortcomings into thematic research clusters."
)

user_prompt = f"""
I am providing you with a complete dataset of {len(limitations_list)} research limitations.
Your goal is to perform a global thematic analysis. 

### THE DATA (Full List):
{json.dumps(limitations_list)}

### YOUR TASK:
1. Review every item in the list above.
2. Group related issues into 5-10 high-level "Topic Clusters."
3. For each cluster, define a professional Topic Name and a detailed Description 
   explaining the technical nature of these combined shortcomings.

### CONSTRAINTS:
- You MUST respond with ONLY a valid JSON object.
- Ensure the description is synthesized from multiple examples in the data.

### OUTPUT FORMAT:
{{
  "topics": [
    {{
      "topic_name": "...",
      "description": "..."
    }}
  ]
}}
"""

# 4. EXECUTE SINGLE-PASS CALL
print(f"Sending all {len(limitations_list)} limitations in one request...")

try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0.3 
    )

    # 5. PARSE AND DISPLAY
    result_content = response.choices[0].message.content
    topic_results = json.loads(result_content)

    # Create a DataFrame for easy viewing/saving
    df_topics = pd.DataFrame(topic_results['topics'])
    
    print("\n✅ Topic Analysis Complete:")
    print("-" * 30)
    print(df_topics)
    
    # Save results to a CSV for your records
    # df_topics.to_csv("synthesized_research_topics.csv", index=False)

except Exception as e:
    print(f"❌ An error occurred: {e}")

Sending all 585 limitations in one request...

✅ Topic Analysis Complete:
------------------------------
                                          topic_name  \
0                    Lack of Novelty and Originality   
1               Insufficient Experimental Validation   
2                       Weak Technical Contributions   
3  Inadequate Literature Review and Contextualiza...   
4                          Limited Scope of Analysis   
5                         Overclaiming Contributions   
6                   Methodological Clarity and Rigor   
7                  Comparative Analysis Deficiencies   
8                           Incremental Improvements   

                                         description  
0  Many papers exhibit a significant lack of nove...  
1  A recurring limitation across several studies ...  
2  Numerous submissions are criticized for presen...  
3  Several papers fail to provide a thorough lite...  
4  Many studies are criticized for having a limit...  
5  A

In [ ]:
df_topics.to_csv("/lstr/sahara/datalab-ml/ibrahim/limagents_update/novelty_llm_agents_rag_based/df_topics_novelty_from_600_nov_gt_gen_by_gpt.csv", index=False)

In [7]:
df_topics

,topic_name,description
0,Lack of Novelty and Originality,Many papers exhibit a significant lack of nove...
1,Insufficient Experimental Validation,A recurring limitation across several studies ...
2,Weak Technical Contributions,Numerous submissions are criticized for presen...
3,Inadequate Literature Review and Contextualiza...,Several papers fail to provide a thorough lite...
4,Limited Scope of Analysis,Many studies are criticized for having a limit...
5,Overclaiming Contributions,A notable issue in several papers is the tende...
6,Methodological Clarity and Rigor,Many critiques focus on the lack of methodolog...
7,Comparative Analysis Deficiencies,A common shortcoming is the inadequate compara...
8,Incremental Improvements,Numerous submissions are characterized by thei...


In [9]:
for i in range(len(df_topics)):
    print(f"Topic {i+1}: {df_topics['topic_name'][i]}")
    print(f"Description: {df_topics['description'][i]}")
    print("-" * 50) 

Topic 1: Lack of Novelty and Originality
Description: Many papers exhibit a significant lack of novelty and originality, often presenting incremental improvements or adaptations of existing methods without introducing new concepts or insights. Common critiques highlight that proposed methods closely resemble prior works, fail to provide substantial advancements, or rely heavily on established techniques, indicating that the contributions may not justify publication.
--------------------------------------------------
Topic 2: Insufficient Experimental Validation
Description: A recurring limitation across several studies is the lack of comprehensive experimental validation. Many papers do not adequately compare their proposed methods with existing benchmarks or fail to demonstrate significant improvements in performance. This raises concerns about the reliability and applicability of the findings, as well as the overall contribution to the field.
-----------------------------------------